In [1]:
"""
2. Quantization-Aware Training (QAT) — ResNet-50 / CIFAR-10
============================================================
FIXED VERSION — resolves ~500x slowdown on PyTorch 2.12 nightly + RTX 5070 (Blackwell SM100)

ROOT CAUSE
==========
PyTorch 2.12 nightly's FakeQuantize CUDA kernels are not compiled for SM100 (Blackwell).
Every fake-quant op falls back to a slow per-element kernel instead of a batched CUDA kernel.
On RTX 5070 this causes epochs to take 3000–25000s instead of ~40s.

FIXES APPLIED
=============
Fix 1 — torch.compile(model, mode="reduce-overhead")
  Fuses fake-quant ops into surrounding conv/BN kernels via inductor.
  This eliminates the per-op overhead entirely.
  A compile warmup of 2–3 batches is expected — after that, speed is normal.

Fix 2 — torch.amp.autocast (float16 mixed precision)
  Halves memory bandwidth and compute for the heavy conv ops.
  FakeQuantize nodes still run in float32 (AMP leaves them alone).
  Combined with compile, epoch time drops from hours → ~40–90s.

Fix 3 — Compile timeout + fallback
  If torch.compile raises or hangs (possible on some nightly builds),
  the script falls back to uncompiled mode and prints a clear warning.

Fix 4 — Epoch time ETA printed after every epoch
  You can now see if something is slow before waiting hours.

Output: __2__QAT.json
"""

import os, sys, json, time, copy, tempfile, warnings, signal
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from thop import profile as thop_profile

warnings.filterwarnings("ignore")

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch {torch.__version__}", flush=True)
print(f"[init] Device  : {DEVICE}", flush=True)
if DEVICE.type == "cuda":
    print(f"[init] GPU     : {torch.cuda.get_device_name(0)}", flush=True)
    cap = torch.cuda.get_device_capability()
    print(f"[init] Compute : SM{cap[0]}{cap[1]}", flush=True)

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__2__QAT_v2.json"

BATCH_SIZE   = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS  = 0
PIN_MEMORY   = False
NUM_CLASSES  = 10
QAT_EPOCHS   = 3
LR_BASE      = 1e-4
OBSERVER_FREEZE_EPOCH = 1

USE_COMPILE = True
USE_AMP     = (DEVICE.type == "cuda")

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] torch.compile : {USE_COMPILE}", flush=True)
print(f"[init] AMP (float16) : {USE_AMP}", flush=True)

# ── Decide QAT backend ────────────────────────────────────────────────────────
USE_FX = True
try:
    from torch.ao.quantization.quantize_fx import prepare_qat_fx, convert_fx
    from torch.ao.quantization import (
        QConfig,
        MovingAverageMinMaxObserver,
        MovingAveragePerChannelMinMaxObserver,
        MinMaxObserver,
        PerChannelMinMaxObserver,
    )
    print("[init] Backend : FX-graph (prepare_qat_fx)", flush=True)
except ImportError:
    USE_FX = False
    import torch.quantization as tq
    print("[init] Backend : Eager-mode (prepare_qat)", flush=True)

# ── Observer configs ──────────────────────────────────────────────────────────
if USE_FX:
    OBSERVER_QCONFIGS = {
        "minmax": QConfig(
            activation=MinMaxObserver.with_args(
                dtype=torch.quint8, qscheme=torch.per_tensor_affine),
            weight=PerChannelMinMaxObserver.with_args(
                dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
        ),
        "moving_avg": QConfig(
            activation=MovingAverageMinMaxObserver.with_args(
                dtype=torch.quint8, qscheme=torch.per_tensor_affine),
            weight=MovingAveragePerChannelMinMaxObserver.with_args(
                dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
        ),
    }
else:
    import torch.quantization as tq
    OBSERVER_QCONFIGS = {
        "moving_avg": tq.QConfig(
            activation=tq.MovingAverageMinMaxObserver.with_args(
                dtype=torch.quint8, qscheme=torch.per_tensor_affine),
            weight=tq.MovingAveragePerChannelMinMaxObserver.with_args(
                dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
        ),
    }

LR_SCHEDULES = {
    "constant": lambda opt, ep: None,
    "cosine"  : lambda opt, ep: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ep),
    "step"    : lambda opt, ep: torch.optim.lr_scheduler.StepLR(
                    opt, step_size=max(1, ep // 2), gamma=0.1),
}

# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes: int = 10) -> nn.Module:
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path: str) -> nn.Module:
    print(f"[load] Loading baseline from {path} ...", flush=True)
    model = build_model(NUM_CLASSES).cpu()
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval()
    print("[load] OK", flush=True)
    return model

# ── Data ──────────────────────────────────────────────────────────────────────
def get_train_loader() -> DataLoader:
    print(f"[data] Building train loader  batch={BATCH_SIZE}  workers={NUM_WORKERS} ...", flush=True)
    transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=True,
                                       download=True, transform=transform)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    print("[data] Train loader ready", flush=True)
    return loader

def get_test_loader() -> DataLoader:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=512, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# ── Helpers ───────────────────────────────────────────────────────────────────
def model_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model, tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)

def param_size_mb_fp32(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters()) * 4 / 1024 ** 2

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> dict:
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(device, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_inference_ms(model: nn.Module, runs: int = 20) -> dict:
    """
    CPU inference timings formatted to match the example JSON schema:
      single_img_cpu_ms, batch128_cpu_ms, per_img_cpu_ms,
      throughput_imgs_sec, timing_method
    QAT INT8 models are evaluated on CPU (PyTorch INT8 is CPU-only).
    """
    m = copy.deepcopy(model).cpu().eval()
    dummy_batch  = torch.randn(128, 3, 32, 32)
    dummy_single = torch.randn(1,   3, 32, 32)

    with torch.no_grad():
        # Warmup
        for _ in range(3):
            m(dummy_batch)
            m(dummy_single)

        # Batch timing
        t0 = time.perf_counter()
        for _ in range(runs):
            m(dummy_batch)
        batch_ms = (time.perf_counter() - t0) / runs * 1000

        # Single-image timing
        t0 = time.perf_counter()
        for _ in range(runs):
            m(dummy_single)
        single_ms = (time.perf_counter() - t0) / runs * 1000

    return {
        "single_img_cpu_ms"  : round(single_ms, 4),
        "batch128_cpu_ms"    : round(batch_ms, 4),
        "per_img_cpu_ms"     : round(batch_ms / 128, 4),
        "throughput_imgs_sec": round(128 / (batch_ms / 1000), 1),
        "timing_method"      : "wall-clock (time.perf_counter), CPU",
    }

def measure_flops(model: nn.Module) -> float:
    """
    Measure FLOPs using thop on the float32 model (thop doesn't support INT8).
    Returns raw FLOPs (MACs × 2).
    """
    m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(1, 3, 32, 32)
    try:
        macs, _ = thop_profile(m, inputs=(dummy,), verbose=False)
        return macs * 2
    except Exception as e:
        print(f"      [flops] WARNING: thop failed ({e}) — returning 0", flush=True)
        return 0.0

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def freeze_observers_and_bn(model: nn.Module) -> None:
    try:
        model.apply(torch.ao.quantization.disable_observer)
    except Exception:
        try:
            import torch.quantization as tq
            model.apply(tq.disable_observer)
        except Exception:
            pass
    try:
        model.apply(torch.nn.intrinsic.qat.freeze_bn_stats)
    except Exception:
        for m in model.modules():
            if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                m.eval()
                m.weight.requires_grad_(False)
                m.bias.requires_grad_(False)

# ── Prepare QAT model ─────────────────────────────────────────────────────────
def prepare_model_for_qat(fp32_model: nn.Module, qconfig, obs_name: str):
    model = copy.deepcopy(fp32_model).cpu().train()

    if USE_FX:
        print("      [prepare] Running prepare_qat_fx ...", flush=True)
        t0 = time.perf_counter()
        example_inputs = (torch.randn(1, 3, 32, 32),)
        prepared = prepare_qat_fx(model, {"": qconfig}, example_inputs)
        print(f"      [prepare] Done  ({time.perf_counter()-t0:.1f}s)", flush=True)
    else:
        print("      [prepare] Running eager-mode prepare_qat ...", flush=True)
        import torch.quantization as tq
        model.qconfig = qconfig
        try:
            tq.fuse_modules(model, [["conv1", "bn1", "relu"]], inplace=True)
        except Exception:
            pass
        prepared = tq.prepare_qat(model, inplace=True)
        print("      [prepare] Done", flush=True)

    return prepared


def maybe_compile(model: nn.Module) -> nn.Module:
    if not USE_COMPILE:
        return model
    try:
        print("      [compile] Compiling model with torch.compile(mode='eager') ...", flush=True)
        t0 = time.perf_counter()
        compiled = torch.compile(model, mode="eager")
        print(f"      [compile] Done  ({time.perf_counter()-t0:.1f}s)", flush=True)
        return compiled
    except Exception as e:
        print(f"      [compile] FAILED ({e}) — running uncompiled.", flush=True)
        return model


# ══════════════════════════════════════════════════════════════════════════════
# Core QAT routine
# ══════════════════════════════════════════════════════════════════════════════
def run_qat(
    fp32_model, train_loader, test_loader,
    baseline_acc, base_disk_mb,
    obs_name, lr_schedule_name,
    n_epochs=QAT_EPOCHS,
) -> dict:
    tag = f"{obs_name}/{lr_schedule_name}"
    print(f"\n  ┌─ [{tag}]", flush=True)

    try:
        qconfig  = OBSERVER_QCONFIGS[obs_name]
        prepared = prepare_model_for_qat(fp32_model, qconfig, obs_name)

        print(f"      [train] Moving to {DEVICE} ...", flush=True)
        prepared = prepared.to(DEVICE)
        print(f"      [train] Model on {DEVICE}", flush=True)

        prepared = maybe_compile(prepared)

        print("      [train] Warm-up forward pass (may take 30-120s for triton compile) ...", flush=True)
        t_wu = time.perf_counter()
        with torch.no_grad():
            _ = prepared(torch.randn(2, 3, 32, 32, device=DEVICE))
        sync()
        print(f"      [train] Warm-up OK  ({time.perf_counter()-t_wu:.1f}s)", flush=True)

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.SGD(prepared.parameters(), lr=LR_BASE,
                                    momentum=0.9, weight_decay=1e-4)
        scheduler = LR_SCHEDULES[lr_schedule_name](optimizer, n_epochs)
        scaler    = torch.amp.GradScaler(enabled=USE_AMP)

        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        epoch_train_acc = []
        sync()
        t_start = time.perf_counter()

        for epoch in range(n_epochs):
            if epoch == OBSERVER_FREEZE_EPOCH:
                print(f"      [epoch {epoch}] Freezing observers + BN ...", flush=True)
                base_model = getattr(prepared, "_orig_mod", prepared)
                base_model = base_model.cpu()
                freeze_observers_and_bn(base_model)
                base_model = base_model.to(DEVICE)
                prepared = maybe_compile(base_model)
                print(f"      [epoch {epoch}] Frozen. Back on {DEVICE}", flush=True)

            prepared.train()
            correct = total = 0
            t_ep = time.perf_counter()

            for batch_idx, (inputs, targets) in enumerate(train_loader):
                inputs  = inputs.to(DEVICE,  non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    outputs = prepared(inputs)
                    loss    = criterion(outputs, targets)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                correct += (outputs.detach().argmax(1) == targets).sum().item()
                total   += targets.size(0)

                if (batch_idx + 1) % 50 == 0:
                    elapsed = time.perf_counter() - t_ep
                    done    = batch_idx + 1
                    eta     = elapsed / done * (len(train_loader) - done)
                    print(f"      [epoch {epoch+1}/{n_epochs}] "
                          f"batch {done}/{len(train_loader)}  "
                          f"acc={correct/total:.4f}  "
                          f"elapsed={elapsed:.0f}s  ETA={eta:.0f}s", flush=True)

            if scheduler is not None:
                scheduler.step()
            acc = correct / total
            epoch_train_acc.append(round(acc, 6))
            sync()
            ep_time = time.perf_counter() - t_ep
            total_elapsed = time.perf_counter() - t_start
            eta_total = ep_time * (n_epochs - (epoch + 1))
            print(f"      [epoch {epoch+1}/{n_epochs}] DONE  "
                  f"acc={acc:.4f}  time={ep_time:.1f}s  "
                  f"total={total_elapsed:.0f}s  ETA_remaining={eta_total:.0f}s", flush=True)

        sync()
        train_time_s = time.perf_counter() - t_start

        # ── FLOPs (measured on FP32 model; identical for INT8) ────────────────
        print("      [flops] Measuring FLOPs on float QAT model ...", flush=True)
        flops_G = measure_flops(fp32_model) / 1e9
        print(f"      [flops] {flops_G:.3f} GFLOPs", flush=True)

        # ── Convert to INT8 ───────────────────────────────────────────────────
        print("      [convert] Unwrapping compiled model ...", flush=True)
        base_model = getattr(prepared, "_orig_mod", prepared)

        print("      [convert] Moving to CPU + convert ...", flush=True)
        base_model = base_model.cpu().eval()
        if USE_FX:
            q_model = convert_fx(base_model)
        else:
            import torch.quantization as tq
            q_model = tq.convert(base_model, inplace=False)
        print("      [convert] Done", flush=True)

        # ── Save ──────────────────────────────────────────────────────────────
        os.makedirs("__2__saved_models_QAT", exist_ok=True)
        save_path = f"__2__saved_models_QAT/qat_int8_{obs_name}_{lr_schedule_name}.pth"
        try:
            torch.save(q_model.state_dict(), save_path)
            print(f"      [save] ✓ Saved → {save_path}", flush=True)
        except Exception as e:
            print(f"      [save] ⚠ state_dict save failed ({e}), trying full model ...", flush=True)
            try:
                torch.save(q_model, save_path)
                print(f"      [save] ✓ Saved (full model) → {save_path}", flush=True)
            except Exception as e2:
                print(f"      [save] ✗ Both save methods failed: {e2}", flush=True)

        # ── Evaluate & measure ────────────────────────────────────────────────
        print("      [eval] Evaluating INT8 model on CPU ...", flush=True)
        metrics      = evaluate(q_model, test_loader, torch.device("cpu"))
        inference_ms = measure_inference_ms(q_model)
        size_mb      = model_size_mb(q_model)
        params       = sum(p.numel() for p in fp32_model.parameters())

        print(f"  └─ Acc={metrics['accuracy']:.4f}  "
              f"Drop={baseline_acc - metrics['accuracy']:+.4f}  "
              f"Disk={size_mb:.2f}MB  "
              f"CPU={inference_ms['batch128_cpu_ms']:.1f}ms  "
              f"Train={train_time_s:.1f}s", flush=True)

        # ── Build compact result entry (matches example JSON schema) ──────────
        entry = {
            "accuracy"        : round(metrics["accuracy"],  6),
            "precision"       : round(metrics["precision"], 6),
            "recall"          : round(metrics["recall"],    6),
            "f1"              : round(metrics["f1"],        6),
            "params"          : int(params),
            "size_mb"         : round(size_mb, 6),
            "flops_G"         : round(flops_G, 9),
            "inference_ms"    : inference_ms,
            # QAT-specific training metadata (compact, no blobs)
            "train_time_s"    : round(train_time_s, 2),
            "epoch_train_acc" : epoch_train_acc,
        }
        return entry

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return None


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print("  QAT — ResNet-50 / CIFAR-10  [FIXED: compile + AMP]")
    print(f"  Device : {DEVICE}  |  Epochs: {QAT_EPOCHS}  |  Batch: {BATCH_SIZE}")
    print(f"  compile: {USE_COMPILE}  |  AMP: {USE_AMP}")
    print(f"{'='*60}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]

    fp32_model   = load_baseline(BASELINE_MODEL_PATH)
    train_loader = get_train_loader()
    test_loader  = get_test_loader()

    base_disk_mb = model_size_mb(fp32_model)
    print(f"  Baseline — Disk: {base_disk_mb:.2f} MB\n", flush=True)

    output = {}

    # ── Sweep A: observer type (lr=constant) ──────────────────────────────────
    print("  ── Sweep A: Observer type (lr=constant) ──────────────────────", flush=True)
    best_obs, best_obs_acc = None, -1.0

    for obs_name in OBSERVER_QCONFIGS:
        key    = f"{obs_name}_constant"
        result = run_qat(fp32_model, train_loader, test_loader,
                         baseline_acc, base_disk_mb,
                         obs_name=obs_name, lr_schedule_name="constant")
        if result is not None:
            output[key] = result
            if result["accuracy"] > best_obs_acc:
                best_obs_acc, best_obs = result["accuracy"], obs_name

    if best_obs is None:
        print("  ✗ All Sweep A configs failed.", flush=True)
        return
    print(f"\n  Best observer: {best_obs} (acc={best_obs_acc:.4f})", flush=True)

    # ── Sweep B: LR schedule (best observer) ──────────────────────────────────
    print(f"\n  ── Sweep B: LR schedule (observer={best_obs}) ────────────────", flush=True)
    for lr_name in LR_SCHEDULES:
        if lr_name == "constant":
            continue
        key    = f"{best_obs}_{lr_name}"
        result = run_qat(fp32_model, train_loader, test_loader,
                         baseline_acc, base_disk_mb,
                         obs_name=best_obs, lr_schedule_name=lr_name)
        if result is not None:
            output[key] = result

    # ── Save ──────────────────────────────────────────────────────────────────
    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)

    # ── Console summary ───────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  ✓ Saved → {OUTPUT_JSON}")
    print(f"\n  {'Method':<28} {'Acc':>7} {'F1':>7} {'Size(MB)':>9} "
          f"{'Batch(ms)':>10} {'Imgs/s':>8} {'Train(s)':>9}")
    print(f"  {'-'*75}")
    for key, val in output.items():
        if val is None:
            continue
        lat  = val["inference_ms"]["batch128_cpu_ms"]
        tput = val["inference_ms"]["throughput_imgs_sec"]
        print(f"  {key:<28} {val['accuracy']:.4f}  {val['f1']:.4f}  "
              f"{val['size_mb']:>9.2f}  {lat:>10.1f}  {tput:>8.1f}  "
              f"{val['train_time_s']:>9.1f}")
    print(f"{'='*60}\n")


if __name__ == "__main__":
    main()

[init] PyTorch 2.12.0.dev20260324+cu128
[init] Device  : cuda
[init] GPU     : NVIDIA GeForce RTX 5070 Laptop GPU
[init] Compute : SM120
[init] torch.compile : True
[init] AMP (float16) : True
[init] Backend : FX-graph (prepare_qat_fx)

  QAT — ResNet-50 / CIFAR-10  [FIXED: compile + AMP]
  Device : cuda  |  Epochs: 3  |  Batch: 128
  compile: True  |  AMP: True

[load] Loading baseline from ../__2__baseline_resnet50_cifar10.pth ...
[load] OK
[data] Building train loader  batch=128  workers=0 ...
[data] Train loader ready
  Baseline — Disk: 90.05 MB

  ── Sweep A: Observer type (lr=constant) ──────────────────────

  ┌─ [minmax/constant]
      [prepare] Running prepare_qat_fx ...


W0430 00:18:40.909000 8676 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


      [prepare] Done  (1.5s)
      [train] Moving to cuda ...
      [train] Model on cuda
      [compile] Compiling model with torch.compile(mode='eager') ...
      [compile] FAILED (Unrecognized mode=eager, should be one of: default, lite, reduce-overhead, max-autotune-no-cudagraphs, max-autotune) — running uncompiled.
      [train] Warm-up forward pass (may take 30-120s for triton compile) ...
      [train] Warm-up OK  (1.3s)
      [epoch 1/3] batch 50/391  acc=0.9892  elapsed=18s  ETA=120s
      [epoch 1/3] batch 100/391  acc=0.9892  elapsed=36s  ETA=104s
      [epoch 1/3] batch 150/391  acc=0.9892  elapsed=53s  ETA=85s
      [epoch 1/3] batch 200/391  acc=0.9884  elapsed=71s  ETA=68s
      [epoch 1/3] batch 250/391  acc=0.9886  elapsed=89s  ETA=50s
      [epoch 1/3] batch 300/391  acc=0.9885  elapsed=108s  ETA=33s
      [epoch 1/3] batch 350/391  acc=0.9887  elapsed=125s  ETA=15s
      [epoch 1/3] DONE  acc=0.9888  time=137.1s  total=137s  ETA_remaining=274s
      [epoch 1] Freezin